# 在 Google Colab（GPU）上訓練 PPO

全部用 2022 年以前的版本：Python 3.10.8、PyTorch 1.13.1（CUDA 11.7）、Stable Baselines3 1.6.2、gym 0.21.0。

1. **執行階段 → 變更執行階段類型 → T4 GPU**
2. 把 `vertical-shmup.zip` 上傳到 Google Drive 根目錄（`MyDrive/`）
3. 依序執行每個 cell。模型、log、報告、影片都存在 `MyDrive/shmup/`，斷線也不會不見。

Colab 內建的 Python 是 3.12，2022 年的套件裝不上去，所以第 3 個 cell 會另外裝一個 Python 3.10.8（約 3–5 分鐘）。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/shmup'
!mkdir -p $DRIVE

In [ ]:
!rm -rf /content/vertical-shmup
!unzip -q -o /content/drive/MyDrive/vertical-shmup.zip -d /content
%cd /content/vertical-shmup

## 安裝 2022 年的環境
micromamba 只是用來下載 Python 3.10.8 的安裝工具；之後所有套件的版本都固定在 `requirements.txt`。

In [ ]:
PY = '/content/py310/bin/python'
!wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba
!/content/bin/micromamba create -y -q -p /content/py310 -c conda-forge python=3.10.8 pip=22.3.1 setuptools=65.5.0 wheel=0.38.4
!{PY} -m pip install -q --no-build-isolation gym==0.21.0
!{PY} -m pip install -q -r requirements.txt "torch==1.13.1+cu117" --extra-index-url https://download.pytorch.org/whl/cu117
!{PY} -m pip install -q --no-deps -e .
!{PY} -c "import torch, stable_baselines3 as sb3, gym, platform; print('python', platform.python_version(), '| torch', torch.__version__, '| sb3', sb3.__version__, '| gym', gym.__version__, '| GPU', torch.cuda.get_device_name(0))"

## 訓練（第一次和斷線後都執行這一格）
`--resume auto`：`MyDrive/shmup/ppo/` 裡有上次的存檔就接著訓練，沒有就從頭開始。玩滿 5000 回合自動停止，
並在第 100 / 500 / 2000 / 5000 回合各存一份模型，同時自動存下紀錄（`milestones.json`）和網頁版模型（`ppo/web/`）。
斷線的話，重新執行第 1、2、3 格（安裝那格）再執行這格。

In [ ]:
!{PY} -u -m shmup train --device cuda --out-dir $DRIVE/ppo --resume auto 2>&1 | tee -a $DRIVE/train.log

## 報告：學習曲線 + 結果表

In [ ]:
!{PY} -m shmup report --run-dir $DRIVE/ppo
from IPython.display import Image, Markdown, display
display(Markdown(open(DRIVE + '/ppo/report/results.md').read().replace('![學習曲線](learning_curve.png)', '')))
display(Image(DRIVE + '/ppo/report/learning_curve.png'))

## 錄影：第 100 / 500 / 2000 / 5000 回合的模型（Colab 有內建 ffmpeg）

In [ ]:
!{PY} -m shmup record --run-dir $DRIVE/ppo --out $DRIVE/videos
!ls -la $DRIVE/videos

## 下載網頁版模型和結果
訓練時已經自動匯出到 `MyDrive/shmup/ppo/web/`。把裡面的檔案放到專案的 `web/model/`，網頁上的選單就能選。

In [ ]:
!cd $DRIVE/ppo && zip -qr $DRIVE/web_models.zip web
from google.colab import files
for f in ['web_models.zip', 'ppo/milestones.json', 'videos/compare_2x2.mp4', 'ppo/report/results.md', 'ppo/report/learning_curve.png']:
    files.download(DRIVE + '/' + f)